# Whitrum AI - Kaggle TPU Training
Founder: Oguzhan (Dr0xy-Drawn)
~350M parameter model

In [ ]:
import os
os.environ['COLAB_TPU_ADDR'] = ''

try:
    import torch_xla
    print('TPU detected')
except:
    print('No TPU, using GPU')

In [ ]:
!pip install -q torch transformers accelerate datasets tokenizers safetensors

In [ ]:
import json
from datasets import Dataset

with open('/kaggle/input/whitrum-data/training_60k.jsonl') as f:
    data = [json.loads(line) for line in f]

ds = Dataset.from_list(data)
print(f'Dataset: {len(ds)} rows')

In [ ]:
import sys
sys.path.insert(0, '/kaggle/input/whitrum-code/WithrumAI')

from whitrum import WhitrumConfig, WhitrumForCausalLM

config = WhitrumConfig(
    vocab_size=60006,
    hidden_size=512,
    intermediate_size=2048,
    num_hidden_layers=12,
    num_attention_heads=8,
    num_key_value_heads=2,
    max_position_embeddings=4096,
)

model = WhitrumForCausalLM(config)
print(f'Params: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M')

In [ ]:
from transformers import AutoTokenizer

try:
    tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-0.5B', trust_remote_code=True)
except:
    tokenizer = AutoTokenizer.from_pretrained('gpt2')

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=512, padding='max_length')

In [ ]:
tokenized_ds = ds.map(tokenize_function, batched=True, remove_columns=['text'])
print(f'Tokenized: {len(tokenized_ds)} rows')

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=['q_proj','k_proj','v_proj','o_proj'],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

args = TrainingArguments(
    output_dir='./whitrum-output',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=200,
    logging_steps=50,
    save_steps=500,
    save_total_limit=3,
    fp16=True,
    optim='adamw_torch',
    report_to='none',
    dataloader_num_workers=2,
)

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds,
    data_collator=collator,
)

trainer.train()

In [ ]:
model.save_pretrained('./whitrum-350m-lora')
tokenizer.save_pretrained('./whitrum-350m-lora')
print('LoRA saved!')

In [ ]:
from peft import PeftModel

base_model = WhitrumForCausalLM(config)
full_model = PeftModel.from_pretrained(base_model, './whitrum-350m-lora')
full_model = full_model.merge_and_unload()
full_model.save_pretrained('./whitrum-350m-full')
tokenizer.save_pretrained('./whitrum-350m-full')
print('Full model saved!')

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/whitrum-350m', 'zip', './whitrum-350m-full')
print('ZIP created!')